# Inferencia en TEST + métricas — EfficientNet-B0 (dataset privado)

Orquesta: (1) inferencia del mejor `.pth` (EfficientNet-B0 meta-prototipos) sobre las **26 imágenes
de test** (12 normal / 14 glaucoma), y (2) cálculo de métricas con **Bootstrap estratificado** +
**Clopper-Pearson**.

- Por imagen: `y_true` (0=normal, 1=glaucoma), `P(glaucoma)` (softmax de distancias negativas),
  `y_pred` (1 si `P(glaucoma) > 0.5`).
- Reúsa `inference_on_test.run_inference` y `metrics_generation.compute_all_metrics` (`/Classifier/Inference`).
- **`SEED` fija el remuestreo del bootstrap → replicable.** Clopper-Pearson es exacto (sin seed).
- Guarda `test_metrics.json` (resumen + predicción por imagen) en Drive.

## Celda 0: Bootstrap (repo + EasyFSL + Drive)

In [ ]:
# --- Bootstrap Colab: repo + easyfsl + Drive ---
import os
REPO = "/content/Medgemma_Segmentation_CIARP_2026"
if not os.path.isdir(REPO):
    !GIT_LFS_SKIP_SMUDGE=1 git clone --depth 1 -b Classifier-Implementation https://github.com/TheBug95/Medgemma_Segmentation_CIARP_2026.git {REPO}
%cd {REPO}
!GIT_LFS_SKIP_SMUDGE=1 git pull origin Classifier-Implementation
!pip install -q easyfsl
from google.colab import drive
drive.mount('/content/drive')
import torch; print("GPU:", torch.cuda.is_available())

## Celda 1: Config + DataModule

**Ajusta** `PTH_PATH` (tu mejor `.pth`), `DATASET_DIR`, `SPLIT_JSON` y `RESULTS_DIR`. `SEED` da replicabilidad al bootstrap.

In [ ]:
import sys, os, json
import numpy as np

REPO = "/content/Medgemma_Segmentation_CIARP_2026"
sys.path.insert(0, f"{REPO}/Experiments/Classifier_Selection")   # modules/
sys.path.insert(0, f"{REPO}/Classifier")                          # private_data_module
sys.path.insert(0, f"{REPO}/Classifier/Inference")                # inference_on_test, metrics_generation

from private_data_module import PrivateDataModule
from inference_on_test import run_inference
from metrics_generation import compute_all_metrics, print_report

# ============== CONFIG — AJUSTA ESTAS RUTAS DE TU DRIVE ==============
PTH_PATH    = "/content/drive/MyDrive/CIARP_private/results_meta_nshot/efficientnet_b0/meta/seed_42/model_N12.pth"  # TODO: tu MEJOR .pth
DATASET_DIR = "/content/drive/MyDrive/Todo Dataset"                # imagenes .jpg (+ mascaras)
SPLIT_JSON  = "/content/drive/MyDrive/CIARP_private/split.json"
RESULTS_DIR = "/content/drive/MyDrive/CIARP_private/results_test_metrics"

SEED        = 42        # replicabilidad del bootstrap
N_BOOTSTRAP = 5000
CI          = 0.95
IMAGE_SIZE  = [224, 224]
# ====================================================================

data_module = PrivateDataModule({
    "split_json": SPLIT_JSON, "images_dir": DATASET_DIR, "masks_dir": DATASET_DIR,
    "image_size": IMAGE_SIZE, "batch_size": 16, "seed": SEED, "cache_images": True,
})
print("Distribucion por split:", data_module.get_class_distribution())
print("Test:", len(data_module.splits["test"]), "imagenes")

## Celda 2: Inferencia sobre las imágenes de test

Carga el `.pth` y predice cada imagen. Tabla por imagen: `y_true | P(glaucoma) | y_pred`.

In [ ]:
y_true, y_proba, y_pred, ids = run_inference(
    PTH_PATH, data_module, split="test", threshold=0.5, seed=SEED
)

names = ["normal", "glaucoma"]
print(f"{'image_id':16s}{'y_true':>10s}{'P(glaucoma)':>13s}{'y_pred':>10s}")
print("-" * 49)
for im, t, p, pr in zip(ids, y_true, y_proba, y_pred):
    print(f"{im:16s}{names[t]:>10s}{p:>13.4f}{names[pr]:>10s}")

print(f"\nGT: {int((y_true == 1).sum())} glaucoma / {int((y_true == 0).sum())} normal | "
      f"predichas glaucoma: {int((y_pred == 1).sum())}")

## Celda 3: Métricas — Bootstrap estratificado + Clopper-Pearson

`compute_all_metrics(..., seed=SEED)` → métricas puntuales + IC bootstrap + IC exacto. Guarda el JSON en Drive.

In [ ]:
res = compute_all_metrics(y_true, y_proba, y_pred, n_bootstrap=N_BOOTSTRAP, ci=CI, seed=SEED)
print_report(res)

# Guardar resumen + prediccion por imagen.
os.makedirs(RESULTS_DIR, exist_ok=True)
payload = dict(res)
payload["pth_path"] = PTH_PATH
payload["per_image"] = [
    {"image_id": im, "y_true": int(t), "p_glaucoma": float(p), "y_pred": int(pr)}
    for im, t, p, pr in zip(ids, y_true, y_proba, y_pred)
]
with open(f"{RESULTS_DIR}/test_metrics.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"\nGuardado: {RESULTS_DIR}/test_metrics.json")